# Manual Bind Confirmation Lab

Notebook for implementing and validating manual barcode binding with conflict detection, confirmation-based reassignment, and resolve-flow compatibility.

## 1. Environment Setup and Test Harness

In [ ]:
from copy import deepcopy
from typing import Any


def assert_equal(actual, expected, label=""):
    if actual != expected:
        raise AssertionError(f"{label} expected={expected} actual={actual}")


def assert_true(value, label=""):
    if not value:
        raise AssertionError(f"{label} expected truthy")


State = dict[str, Any]

initial_state: State = {
    "cache": {
        "001": {
            "4601234567890": {"codes": ["A101", "A102"], "source": "backend"},
            "4600000000007": {"codes": ["A777"], "source": "manual"},
            "4821111111111": {"codes": ["A200"], "source": "backend"},
        },
        "002": {
            "4601234567890": {"codes": ["B001"], "source": "backend"},
            "5902222222222": {"codes": ["B777"], "source": "manual"},
        },
    },
    "log": []
}

state = deepcopy(initial_state)
state

## 2. Model Barcode-to-ItemCode Cache

Define structures and helpers for normalization, store scope, and deduplication.

In [ ]:
def normalize_text(value: Any) -> str:
    return str(value or "").strip()


def normalize_codes(record: dict) -> list[str]:
    if not record:
        return []
    raw = record.get("codes")
    if isinstance(raw, list):
        return [normalize_text(x) for x in raw if normalize_text(x)]
    one = normalize_text(record.get("code"))
    return [one] if one else []


def dedupe_codes(codes: list[str]) -> list[str]:
    return sorted(set(normalize_text(x) for x in codes if normalize_text(x)))


def get_store_cache(cache: dict, store_number: str) -> dict:
    return cache.setdefault(normalize_text(store_number), {})


get_store_cache(state["cache"], "001")

## 3. Implement Conflict Lookup for Manual Binding

Find existing barcode links for selected item code in the same store scope.

In [ ]:
def find_conflicts(cache: dict, store_number: str, current_barcode: str, item_code: str) -> dict:
    store_cache = get_store_cache(cache, store_number)
    current_barcode = normalize_text(current_barcode)
    item_code = normalize_text(item_code)

    existing = []
    for barcode, record in store_cache.items():
        if barcode == current_barcode:
            continue
        if item_code in normalize_codes(record):
            existing.append(barcode)

    return {
        "conflict": len(existing) > 0,
        "existingBarcodes": sorted(existing),
        "currentBarcode": current_barcode,
        "itemCode": item_code,
        "storeNumber": normalize_text(store_number)
    }


find_conflicts(state["cache"], "001", "4821111111111", "A101")

## 4. Implement Atomic Rebind Operation

Transactional copy-on-write update:
- remove item code from old barcodes
- add to current barcode
- dedupe
- commit only if successful

In [ ]:
def atomic_rebind(cache: dict, store_number: str, current_barcode: str, item_code: str) -> dict:
    store_number = normalize_text(store_number)
    current_barcode = normalize_text(current_barcode)
    item_code = normalize_text(item_code)

    next_cache = deepcopy(cache)
    store_cache = get_store_cache(next_cache, store_number)

    # Remove from all barcodes in the same store.
    for barcode in list(store_cache.keys()):
        record = store_cache[barcode]
        codes = [c for c in normalize_codes(record) if c != item_code]
        if codes:
            record["codes"] = dedupe_codes(codes)
            store_cache[barcode] = record
        else:
            store_cache.pop(barcode, None)

    current = store_cache.get(current_barcode, {"codes": [], "source": "manual"})
    current_codes = dedupe_codes(normalize_codes(current) + [item_code])
    current["codes"] = current_codes
    current["source"] = "manual"
    store_cache[current_barcode] = current

    return next_cache


preview_cache = atomic_rebind(state["cache"], "001", "4821111111111", "A101")
preview_cache["001"]["4821111111111"]

## 5. Expose `bind-barcode` API Handler

Endpoint-like logic:
- return 409 conflict when existing links present and confirm is absent
- perform atomic reassignment only when `confirm=True`

In [ ]:
from dataclasses import dataclass


@dataclass
class BindRequest:
    barcode: str
    itemCode: str
    storeNumber: str
    confirm: bool = False


def bind_barcode_handler(request: BindRequest, state_obj: State) -> tuple[int, dict, State]:
    barcode = normalize_text(request.barcode)
    item_code = normalize_text(request.itemCode)
    store_number = normalize_text(request.storeNumber)

    if not barcode or not item_code or not store_number:
        return 400, {"ok": False, "error": "barcode, itemCode, storeNumber are required"}, state_obj

    conflict_info = find_conflicts(state_obj["cache"], store_number, barcode, item_code)
    if conflict_info["conflict"] and request.confirm is not True:
        return 409, {
            "ok": False,
            "conflict": True,
            "barcode": barcode,
            "itemCode": item_code,
            "storeNumber": store_number,
            "existingBarcodes": conflict_info["existingBarcodes"],
            "error": f"Код товара уже привязан к штрихкоду {', '.join(conflict_info['existingBarcodes'])}"
        }, state_obj

    next_state = deepcopy(state_obj)
    next_state["cache"] = atomic_rebind(next_state["cache"], store_number, barcode, item_code)
    next_state["log"].append({"event": "bind-barcode", "barcode": barcode, "itemCode": item_code, "confirm": request.confirm})

    return 200, {
        "ok": True,
        "conflict": False,
        "barcode": barcode,
        "itemCode": item_code,
        "storeNumber": store_number,
        "existingBarcodes": conflict_info["existingBarcodes"]
    }, next_state


code, payload, _ = bind_barcode_handler(BindRequest("4821111111111", "A101", "001", False), state)
code, payload

## 6. Client-Side Manual Bind Flow with Confirmation

Pseudo-UI flow:
- call bind endpoint
- on 409 show conflict message
- if user declines, keep state unchanged
- if user confirms, retry with `confirm=True`

In [ ]:
def client_bind_flow(state_obj: State, barcode: str, item_code: str, store_number: str, user_confirms: bool) -> tuple[State, dict]:
    req = BindRequest(barcode=barcode, itemCode=item_code, storeNumber=store_number, confirm=False)
    status, payload, unchanged = bind_barcode_handler(req, state_obj)

    if status != 409:
        return unchanged, {"status": status, "payload": payload, "step": "initial_success"}

    conflict_msg = {
        "status": status,
        "payload": payload,
        "step": "conflict_prompt",
        "uiMessage": f"Код {item_code} уже привязан к {', '.join(payload['existingBarcodes'])}. Перепривязать к {barcode}?"
    }

    if not user_confirms:
        return unchanged, {**conflict_msg, "decision": "cancel"}

    req_confirm = BindRequest(barcode=barcode, itemCode=item_code, storeNumber=store_number, confirm=True)
    status2, payload2, confirmed_state = bind_barcode_handler(req_confirm, unchanged)
    return confirmed_state, {"status": status2, "payload": payload2, "step": "confirmed", "decision": "confirm"}


cancel_state, cancel_info = client_bind_flow(state, "4821111111111", "A101", "001", user_confirms=False)
confirm_state, confirm_info = client_bind_flow(state, "4821111111111", "A101", "001", user_confirms=True)
cancel_info, confirm_info

## 7. Keep Manual Bind Available After Successful Resolve

Simulate successful auto-resolve and keep manual bind controls enabled for explicit override.

In [ ]:
def resolve_barcode_baseline(cache: dict, store_number: str, barcode: str, fallback_item_codes: list[str]) -> dict:
    barcode = normalize_text(barcode)
    store_cache = get_store_cache(cache, store_number)

    local = store_cache.get(barcode)
    if local and normalize_codes(local):
        return {"ok": True, "resolved": True, "barcode": barcode, "codes": dedupe_codes(normalize_codes(local)), "source": local.get("source", "cache")}

    if barcode in set(fallback_item_codes):
        return {"ok": True, "resolved": True, "barcode": barcode, "codes": [barcode], "source": "item-code-fallback"}

    return {"ok": True, "resolved": False, "barcode": barcode, "codes": [], "source": "not-found"}


auto = resolve_barcode_baseline(state["cache"], "001", "4601234567890", ["A101", "A999"])
manual_bind_enabled = auto["resolved"] is True
{"autoResolve": auto, "manualBindEnabled": manual_bind_enabled, "bindTargetBarcode": auto["barcode"]}

## 8. Unit Tests: Reject, Confirm, No-Duplicate Cases

Pytest-style test functions for refusal, confirmed move, duplicate prevention, and idempotency.

In [ ]:
def test_reject_keeps_state_unchanged():
    s0 = deepcopy(initial_state)
    s1, info = client_bind_flow(s0, "4821111111111", "A101", "001", user_confirms=False)
    assert_equal(info["status"], 409, "reject_status")
    assert_equal(s1, s0, "reject_no_mutation")


def test_confirm_moves_code_correctly():
    s0 = deepcopy(initial_state)
    s1, info = client_bind_flow(s0, "4821111111111", "A101", "001", user_confirms=True)
    assert_equal(info["status"], 200, "confirm_status")
    assert_true("A101" in normalize_codes(s1["cache"]["001"]["4821111111111"]), "target_has_code")
    assert_true("A101" not in normalize_codes(s1["cache"]["001"]["4601234567890"]), "old_barcode_removed")


def test_no_duplicates_after_repeated_binding():
    s0 = deepcopy(initial_state)
    s1, _ = client_bind_flow(s0, "4821111111111", "A101", "001", user_confirms=True)
    s2, _ = client_bind_flow(s1, "4821111111111", "A101", "001", user_confirms=True)
    codes = normalize_codes(s2["cache"]["001"]["4821111111111"])
    assert_equal(codes.count("A101"), 1, "dedupe_count")


def test_idempotent_bind_same_target():
    s0 = deepcopy(initial_state)
    _, info1 = client_bind_flow(s0, "4601234567890", "A101", "001", user_confirms=True)
    s2, info2 = client_bind_flow(s0, "4601234567890", "A101", "001", user_confirms=True)
    assert_equal(info1["status"], 200, "same_target_status_first")
    assert_equal(info2["status"], 200, "same_target_status_second")
    assert_equal(normalize_codes(s2["cache"]["001"]["4601234567890"]).count("A101"), 1, "same_target_no_dup")


def run_unit_tests_section_8():
    test_reject_keeps_state_unchanged()
    test_confirm_moves_code_correctly()
    test_no_duplicates_after_repeated_binding()
    test_idempotent_bind_same_target()
    print("Section 8 tests passed")


run_unit_tests_section_8()

## 9. Regression Tests for Existing `resolve-barcode` Flow

Validate cache-first and fallback payload shape remain stable after manual bind logic.

In [ ]:
def test_resolve_cache_first_shape():
    s0 = deepcopy(initial_state)
    out = resolve_barcode_baseline(s0["cache"], "001", "4601234567890", ["A101"])
    assert_equal(out["ok"], True, "cache_ok")
    assert_equal(out["resolved"], True, "cache_resolved")
    assert_equal(out["barcode"], "4601234567890", "cache_barcode")
    assert_true(isinstance(out["codes"], list), "cache_codes_type")


def test_resolve_fallback_shape():
    s0 = deepcopy(initial_state)
    out = resolve_barcode_baseline(s0["cache"], "001", "A404", ["A404", "A999"])
    assert_equal(out, {
        "ok": True,
        "resolved": True,
        "barcode": "A404",
        "codes": ["A404"],
        "source": "item-code-fallback"
    }, "fallback_shape")


def test_resolve_not_found_shape():
    s0 = deepcopy(initial_state)
    out = resolve_barcode_baseline(s0["cache"], "001", "0000000000000", ["A404"])
    assert_equal(out["ok"], True, "not_found_ok")
    assert_equal(out["resolved"], False, "not_found_resolved")
    assert_equal(out["codes"], [], "not_found_codes")


def run_regression_tests_section_9():
    test_resolve_cache_first_shape()
    test_resolve_fallback_shape()
    test_resolve_not_found_shape()
    print("Section 9 regression tests passed")


run_regression_tests_section_9()